# Module 02: Pretrained Encoders for Classification

**ECBS5200 Pre-Work** — Practical Deep Learning Engineering

In this notebook, you'll load a pretrained encoder model, inspect its
architecture, and see exactly what it produces when you feed it a real
consumer complaint. No GPU needed — this runs on CPU.

## Setup

We need `transformers` (for the model and tokenizer) and `datasets`
(to load a real complaint from our course dataset).

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import torch

# Load the tokenizer (same one from Module 01)
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

print(f"Tokenizer loaded: answerdotai/ModernBERT-base")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

## Load the base encoder

`AutoModel` gives us the encoder backbone — no classification head.
It takes token IDs in and produces dense vectors out.

In [ ]:
base_model = AutoModel.from_pretrained("answerdotai/ModernBERT-base")

print(base_model)

That's a lot of output. Don't worry about understanding every layer right
now. The key things to notice:
- There are **multiple transformer layers** (the repeated blocks)
- The hidden size is **768** — that's the dimension of each token's vector

Let's count the parameters.

In [ ]:
total_params = sum(p.numel() for p in base_model.parameters())
print(f"Total parameters in base model: {total_params:,}")
print(f"That's about {total_params / 1e6:.0f} million parameters")

## Load the classification model

`AutoModelForSequenceClassification` wraps the same encoder and adds a
classification head on top. We specify `num_labels=113` — that's the
number of complaint categories we'll use after some label cleanup in Week 1.

In [ ]:
classification_model = AutoModelForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base",
    num_labels=113
)

print(classification_model)

Notice the extra layers at the end — that's the **classification head**.
It has a prediction head (dense layer + GELU activation + LayerNorm)
followed by a final linear layer that maps 768 dimensions to 113 outputs
(one per class).

Let's count parameters and see the difference.

In [ ]:
cls_params = sum(p.numel() for p in classification_model.parameters())
head_params = cls_params - total_params

print(f"Base model parameters:          {total_params:,}")
print(f"Classification model parameters: {cls_params:,}")
print(f"Classification head parameters:  {head_params:,}")
print(f"\nThe head is {head_params / cls_params * 100:.2f}% of total parameters")
print(f"The encoder backbone is {total_params / cls_params * 100:.2f}% of total parameters")

The classification head adds very few parameters compared to the encoder.
Almost all the "intelligence" is in the pretrained backbone.

## Process a real complaint

Let's load one real complaint from the dataset and feed it through both models.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("determined-ai/consumer_complaints_medium", split="train")

# Grab the first complaint
complaint_text = dataset[0]["Consumer Complaint"]
complaint_label = dataset[0]["Issue"]

print(f"Label: {complaint_label}")
print(f"Text:  {complaint_text[:300]}...")

## Tokenize and run through the base model

We tokenize the complaint (just like Module 01), then feed the tokens
into the base encoder.

In [ ]:
# Tokenize
inputs = tokenizer(
    complaint_text,
    max_length=128,
    truncation=True,
    padding="max_length",
    return_tensors="pt"  # return PyTorch tensors
)

print(f"input_ids shape:      {inputs['input_ids'].shape}")
print(f"attention_mask shape: {inputs['attention_mask'].shape}")
print(f"Real tokens (non-padding): {inputs['attention_mask'].sum().item()}")

In [ ]:
# Run through the base model (no gradient computation needed)
with torch.no_grad():
    base_output = base_model(**inputs)

print(f"Output type: {type(base_output)}")
print(f"Last hidden state shape: {base_output.last_hidden_state.shape}")

The shape is `[1, 128, 768]`:
- **1** — one complaint (batch size 1)
- **128** — one vector per token position (our max_length)
- **768** — each vector has 768 dimensions

Every token gets its own 768-dimensional representation.

## Extract the [CLS] representation

The [CLS] token is at position 0. Its vector is the "summary" of the
entire complaint — and this is what gets fed to the classification head.

In [ ]:
cls_vector = base_output.last_hidden_state[0, 0, :]  # batch 0, token 0, all 768 dims

print(f"[CLS] vector shape: {cls_vector.shape}")
print(f"[CLS] vector (first 10 values): {cls_vector[:10]}")
print(f"\nThese 768 numbers summarize the entire complaint.")
print(f"The classification head maps these to 113 class scores.")

## Run through the classification model

Now let's feed the same tokens into the classification model and see
the difference.

In [ ]:
with torch.no_grad():
    cls_output = classification_model(**inputs)

print(f"Output type: {type(cls_output)}")
print(f"Logits shape: {cls_output.logits.shape}")

The shape is `[1, 113]`:
- **1** — one complaint
- **113** — one score (logit) per class

Instead of 768-dim vectors for each token, we get 113 numbers — one
score for each complaint category.

In [ ]:
logits = cls_output.logits[0]

print(f"Logits (all 113 values):")
print(logits)
print(f"\nHighest logit at index: {logits.argmax().item()}")
print(f"Highest logit value:    {logits.max().item():.4f}")

These logits are essentially random right now — the classification head
was initialized with random weights. The model hasn't been fine-tuned yet,
so it has no idea how to classify complaints. That's what training
(fine-tuning) will do: teach the head to produce the right scores.

## Summary: base model vs. classification model

| | Base Model | Classification Model |
|:---|:---|:---|
| **Loaded with** | `AutoModel` | `AutoModelForSequenceClassification` |
| **Output** | 768-dim vector per token | 113 logits (one per class) |
| **Output shape** | `[batch, seq_len, 768]` | `[batch, 113]` |
| **Has classification head?** | No | Yes |
| **Can make predictions?** | No (just embeddings) | Yes (after fine-tuning) |

## Check yourself

Before moving to Module 03, make sure you can answer these:

1. What does "pretrained" mean? What did the model learn during pretraining?
2. What is the shape of the base model's output, and what does each
   dimension represent?
3. What is the [CLS] token's role in classification?
4. How many parameters does the classification head add compared to the
   full encoder? Why is it so small?
5. Why are the classification model's logits essentially random before
   fine-tuning?